<a href="https://colab.research.google.com/github/louistrue/DB-1/blob/main/DB1_W4_Dateien_Lesen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📁 Digitales Bauen 1: Daten aus Dateien lesen
## Woche 4 – CSV und JSON in Python

**Berner Fachhochschule** | Architektur, Holz und Bau | BSc Bauingenieurwesen

---

In diesem Notebook lest ihr die **Velobücke** aus einer Datei ein und arbeitet damit – genau wie in Woche 3, aber die Daten kommen jetzt nicht mehr aus eurem Code, sondern aus einer Datei.

---
## Schritt 1: Dateien erstellen

Zuerst erstellen wir die Velobücke als CSV- und JSON-Datei direkt im Colab. In der Praxis würdet ihr diese Dateien von einem Planer bekommen oder aus einer Software exportieren.

In [1]:
# Velobruecke als CSV-Datei erstellen

csv_inhalt = """id,typ,material,b_cm,h_cm,laenge_m,profil
S1a,Stuetze,C25/30,30,30,3.20,
S2a,Stuetze,C30/37,40,40,3.20,
S3a,Stuetze,C25/30,30,30,3.20,
S1b,Stuetze,C25/30,30,30,3.20,
S2b,Stuetze,C30/37,40,40,3.20,
S3b,Stuetze,C25/30,30,30,3.20,
T1a,Traeger,S355,,,,HEB 300
T1b,Traeger,S355,,,,HEB 300
T2a,Traeger,S355,,,,HEB 340
T2b,Traeger,S355,,,,HEB 340
P1,Platte,C30/37,400,25,24.00,"""

with open("bruecke.csv", "w") as f:
    f.write(csv_inhalt)

print("bruecke.csv erstellt!")

bruecke.csv erstellt!


In [2]:
# Velobruecke als JSON-Datei erstellen

import json

bruecke_json = [
    # Stuetzenreihe links (y=0.8m): unter Traeger-Linie 1
    {"id": "S1a", "typ": "Stuetze", "material": "C25/30", "b_cm": 30, "h_cm": 30, "laenge_m": 3.20},
    {"id": "S2a", "typ": "Stuetze", "material": "C30/37", "b_cm": 40, "h_cm": 40, "laenge_m": 3.20},
    {"id": "S3a", "typ": "Stuetze", "material": "C25/30", "b_cm": 30, "h_cm": 30, "laenge_m": 3.20},
    # Stuetzenreihe rechts (y=3.2m): unter Traeger-Linie 2
    {"id": "S1b", "typ": "Stuetze", "material": "C25/30", "b_cm": 30, "h_cm": 30, "laenge_m": 3.20},
    {"id": "S2b", "typ": "Stuetze", "material": "C30/37", "b_cm": 40, "h_cm": 40, "laenge_m": 3.20},
    {"id": "S3b", "typ": "Stuetze", "material": "C25/30", "b_cm": 30, "h_cm": 30, "laenge_m": 3.20},
    # Traeger Feld 1 (Spannweite 10m): x=0..10
    {"id": "T1a", "typ": "Traeger", "material": "S355", "profil": "HEB 300", "laenge_m": 10.00},
    {"id": "T1b", "typ": "Traeger", "material": "S355", "profil": "HEB 300", "laenge_m": 10.00},
    # Traeger Feld 2 (Spannweite 14m): x=10..24
    {"id": "T2a", "typ": "Traeger", "material": "S355", "profil": "HEB 340", "laenge_m": 14.00},
    {"id": "T2b", "typ": "Traeger", "material": "S355", "profil": "HEB 340", "laenge_m": 14.00},
    # Fahrbahnplatte
    {"id": "P1", "typ": "Platte", "material": "C30/37", "b_cm": 400, "h_cm": 25, "laenge_m": 24.00},
]

with open("bruecke.json", "w") as f:
    json.dump(bruecke_json, f, indent=2, ensure_ascii=False)

print(f"bruecke.json erstellt! ({len(bruecke_json)} Elemente)")

bruecke.json erstellt! (11 Elemente)


---
## Schritt 2: CSV einlesen

`csv.DictReader` macht aus jeder Zeile ein Dictionary. Die Spaltennamen werden zu Keys.

In [3]:
import csv

with open("bruecke.csv") as f:
    reader = csv.DictReader(f)
    csv_daten = list(reader)

# Was haben wir?
print(f"Anzahl Elemente: {len(csv_daten)}")
print()

for element in csv_daten:
    print(element["id"], element["typ"], element["material"])

Anzahl Elemente: 11

S1a Stuetze C25/30
S2a Stuetze C30/37
S3a Stuetze C25/30
S1b Stuetze C25/30
S2b Stuetze C30/37
S3b Stuetze C25/30
T1a Traeger S355
T1b Traeger S355
T2a Traeger S355
T2b Traeger S355
P1 Platte C30/37


**Das Problem mit CSV:** Alle Werte sind Strings – auch Zahlen!

In [4]:
# CSV: Alles ist ein String
element = csv_daten[0]   # S1

print(f"b_cm = {element['b_cm']}")
print(f"Typ:   {type(element['b_cm'])}")
print()

# Das geht schief:
try:
    flaeche = element["b_cm"] * element["h_cm"]
except TypeError as e:
    print(f"Fehler: {e}")
    print("Man kann Strings nicht multiplizieren!")
    print()
    print("Loesung: int() oder float() verwenden")
    flaeche = int(element["b_cm"]) * int(element["h_cm"])
    print(f"Flaeche: {flaeche} cm2")

b_cm = 30
Typ:   <class 'str'>

Fehler: can't multiply sequence by non-int of type 'str'
Man kann Strings nicht multiplizieren!

Loesung: int() oder float() verwenden
Flaeche: 900 cm2


---
## Schritt 3: JSON einlesen

`json.load()` gibt direkt Python-Dicts zurück – **mit den richtigen Datentypen**.

In [5]:
import json

with open("bruecke.json") as f:
    daten = json.load(f)

# Was haben wir?
print(f"Anzahl Elemente: {len(daten)}")
print()

for element in daten:
    print(element["id"], element["typ"], element["material"])

Anzahl Elemente: 11

S1a Stuetze C25/30
S2a Stuetze C30/37
S3a Stuetze C25/30
S1b Stuetze C25/30
S2b Stuetze C30/37
S3b Stuetze C25/30
T1a Traeger S355
T1b Traeger S355
T2a Traeger S355
T2b Traeger S355
P1 Platte C30/37


In [6]:
# JSON: Datentypen stimmen sofort
element = daten[0]   # S1

print(f"b_cm = {element['b_cm']}")
print(f"Typ:   {type(element['b_cm'])}")
print()

# Das funktioniert direkt!
flaeche = element["b_cm"] * element["h_cm"]
print(f"Flaeche: {flaeche} cm2")
print()
print("Kein int() noetig. JSON behaelt die Datentypen.")

b_cm = 30
Typ:   <class 'int'>

Flaeche: 900 cm2

Kein int() noetig. JSON behaelt die Datentypen.


---
## Schritt 4: Was man mit strukturierten Daten machen kann

Jetzt wird es spannend. Die Daten sind geladen. In wenigen Zeilen Code könnt ihr **filtern, berechnen, prüfen und auswerten** – automatisiert, für beliebig viele Elemente.

### 4.1 Filtern: Alle Stützen finden

In [7]:
stuetzen = [e for e in daten if e["typ"] == "Stuetze"]

print(f"{len(stuetzen)} Stuetzen gefunden:")
for s in stuetzen:
    print(f"  {s['id']}: {s['b_cm']}x{s['h_cm']} cm, {s['material']}, L={s['laenge_m']} m")

6 Stuetzen gefunden:
  S1a: 30x30 cm, C25/30, L=3.2 m
  S2a: 40x40 cm, C30/37, L=3.2 m
  S3a: 30x30 cm, C25/30, L=3.2 m
  S1b: 30x30 cm, C25/30, L=3.2 m
  S2b: 40x40 cm, C30/37, L=3.2 m
  S3b: 30x30 cm, C25/30, L=3.2 m


### 4.2 Berechnen: Betonvolumen aller Stützen

In [8]:
print("Betonvolumen pro Stuetze:")
print()

gesamt = 0
for s in stuetzen:
    vol = s["b_cm"] * s["h_cm"] * (s["laenge_m"] * 100) / 1_000_000  # m3
    gesamt += vol
    print(f"  {s['id']}: {s['b_cm']}x{s['h_cm']} cm, L={s['laenge_m']} m  ->  {vol:.3f} m3")

print(f"\nGesamtvolumen Beton (Stuetzen): {gesamt:.3f} m3")
print(f"Betongewicht (25 kN/m3):         {gesamt * 25:.1f} kN")
print(f"Betonkosten (CHF 180/m3):        CHF {gesamt * 180:.0f}")

Betonvolumen pro Stuetze:

  S1a: 30x30 cm, L=3.2 m  ->  0.288 m3
  S2a: 40x40 cm, L=3.2 m  ->  0.512 m3
  S3a: 30x30 cm, L=3.2 m  ->  0.288 m3
  S1b: 30x30 cm, L=3.2 m  ->  0.288 m3
  S2b: 40x40 cm, L=3.2 m  ->  0.512 m3
  S3b: 30x30 cm, L=3.2 m  ->  0.288 m3

Gesamtvolumen Beton (Stuetzen): 2.176 m3
Betongewicht (25 kN/m3):         54.4 kN
Betonkosten (CHF 180/m3):        CHF 392


### 4.3 Prüfen: Plausibilitätskontrolle

In [9]:
print("Plausibilitaetspruefung:")
print()

for e in daten:
    probleme = []

    # Laenge pruefen
    if e["laenge_m"] <= 0:
        probleme.append(f"Laenge ungueltig: {e['laenge_m']} m")
    if e["laenge_m"] > 15:
        probleme.append(f"Laenge ungewoehnlich gross: {e['laenge_m']} m")

    # Stuetzen-Querschnitt pruefen
    if e["typ"] == "Stuetze":
        if e["b_cm"] < 20:
            probleme.append(f"Breite sehr klein: {e['b_cm']} cm")
        if e["laenge_m"] > 4.0:
            probleme.append(f"Stuetze ueber 4m: Knicken pruefen!")
        # Schlankheit
        h_ratio = (e["laenge_m"] * 100) / e["b_cm"]
        if h_ratio > 15:
            probleme.append(f"Schlankheit {h_ratio:.0f} > 15: kritisch!")

    if probleme:
        print(f"  {e['id']} ({e['typ']})")
        for p in probleme:
            print(f"    -> {p}")
    else:
        print(f"  {e['id']} ({e['typ']}): OK")

Plausibilitaetspruefung:

  S1a (Stuetze): OK
  S2a (Stuetze): OK
  S3a (Stuetze): OK
  S1b (Stuetze): OK
  S2b (Stuetze): OK
  S3b (Stuetze): OK
  T1a (Traeger): OK
  T1b (Traeger): OK
  T2a (Traeger): OK
  T2b (Traeger): OK
  P1 (Platte)
    -> Laenge ungewoehnlich gross: 24.0 m


### 4.4 Auswerten: Materialauszug

In [10]:
# Wie viele Elemente pro Material?
materialien = {}
for e in daten:
    mat = e["material"]
    if mat not in materialien:
        materialien[mat] = []
    materialien[mat].append(e["id"])

print("Materialauszug:")
print()
for mat, elemente in materialien.items():
    print(f"  {mat}: {len(elemente)} Elemente ({', '.join(elemente)})")

Materialauszug:

  C25/30: 4 Elemente (S1a, S3a, S1b, S3b)
  C30/37: 3 Elemente (S2a, S2b, P1)
  S355: 4 Elemente (T1a, T1b, T2a, T2b)


### 4.5 Visualisieren: Querschnitte vergleichen

In [11]:
# Einfaches Balkendiagramm mit reinem Python (kein Import noetig)

stuetzen_sorted = sorted(stuetzen, key=lambda s: s["b_cm"] * s["h_cm"])

print("Querschnittsflaechen der Stuetzen:")
print()

max_flaeche = max(s["b_cm"] * s["h_cm"] for s in stuetzen)

for s in stuetzen_sorted:
    flaeche = s["b_cm"] * s["h_cm"]
    balken = "=" * int(flaeche / max_flaeche * 40)
    print(f"  {s['id']}  {s['b_cm']:>3}x{s['h_cm']:<3} cm  {balken} {flaeche} cm2")

Querschnittsflaechen der Stuetzen:

  S1a   30x30  cm  ====================== 900 cm2
  S3a   30x30  cm  ====================== 900 cm2
  S1b   30x30  cm  ====================== 900 cm2
  S3b   30x30  cm  ====================== 900 cm2
  S2a   40x40  cm  ======================================== 1600 cm2
  S2b   40x40  cm  ======================================== 1600 cm2


---
## Schritt 5: Ergebnisse speichern

Der Round-Trip: Datei einlesen → verarbeiten → Ergebnis als neue Datei speichern.

In [12]:
# Ergebnisse berechnen und als neue JSON-Datei speichern

ergebnisse = []

for e in daten:
    ergebnis = {"id": e["id"], "typ": e["typ"], "material": e["material"]}

    if e["typ"] == "Stuetze":
        vol = e["b_cm"] * e["h_cm"] * (e["laenge_m"] * 100) / 1_000_000
        ergebnis["volumen_m3"] = round(vol, 4)
        ergebnis["gewicht_kn"] = round(vol * 25, 1)
        A = e["b_cm"] * e["h_cm"]
        ergebnis["querschnitt_cm2"] = A

    ergebnis["laenge_m"] = e["laenge_m"]
    ergebnisse.append(ergebnis)

# Speichern
with open("bruecke_auswertung.json", "w") as f:
    json.dump(ergebnisse, f, indent=2, ensure_ascii=False)

print("bruecke_auswertung.json gespeichert!")
print()

# Anzeigen
for e in ergebnisse:
    print(json.dumps(e, ensure_ascii=False))

bruecke_auswertung.json gespeichert!

{"id": "S1a", "typ": "Stuetze", "material": "C25/30", "volumen_m3": 0.288, "gewicht_kn": 7.2, "querschnitt_cm2": 900, "laenge_m": 3.2}
{"id": "S2a", "typ": "Stuetze", "material": "C30/37", "volumen_m3": 0.512, "gewicht_kn": 12.8, "querschnitt_cm2": 1600, "laenge_m": 3.2}
{"id": "S3a", "typ": "Stuetze", "material": "C25/30", "volumen_m3": 0.288, "gewicht_kn": 7.2, "querschnitt_cm2": 900, "laenge_m": 3.2}
{"id": "S1b", "typ": "Stuetze", "material": "C25/30", "volumen_m3": 0.288, "gewicht_kn": 7.2, "querschnitt_cm2": 900, "laenge_m": 3.2}
{"id": "S2b", "typ": "Stuetze", "material": "C30/37", "volumen_m3": 0.512, "gewicht_kn": 12.8, "querschnitt_cm2": 1600, "laenge_m": 3.2}
{"id": "S3b", "typ": "Stuetze", "material": "C25/30", "volumen_m3": 0.288, "gewicht_kn": 7.2, "querschnitt_cm2": 900, "laenge_m": 3.2}
{"id": "T1a", "typ": "Traeger", "material": "S355", "laenge_m": 10.0}
{"id": "T1b", "typ": "Traeger", "material": "S355", "laenge_m": 10.0}
{"id": "

**Das ist der Punkt:** Ihr habt gerade eine Datei eingelesen, Berechnungen durchgeführt und die Ergebnisse als neue Datei gespeichert. Automatisiert, nachvollziehbar, reproduzierbar.

Stellt euch vor, die Velobücke hat nicht 6 Elemente, sondern 600. Der Code bleibt derselbe.

---
## 🚀 Bonus: 3D-Visualisierung der Velobücke

Aus denselben JSON-Daten, die ihr gerade eingelesen habt, bauen wir jetzt ein **vollständiges 3D-Modell** der Velobücke. Ihr könnt es drehen, zoomen und von allen Seiten betrachten.

Das ist ein Vorgeschmack auf Woche 6, wo ihr genau das systematisch aufbaut.

In [13]:
import plotly.graph_objects as go
import numpy as np
import json

with open("bruecke.json") as f:
    daten = json.load(f)

# ============================================================
# Brueckengeometrie
# ============================================================
#
#   Draufsicht (x-y):
#
#    y=3.2  S1b ----T1b---- S2b --------T2b-------- S3b
#            |    10m Feld 1  |      14m Feld 2       |
#    y=0.8  S1a ----T1a---- S2a --------T2a-------- S3a
#           x=0             x=10                    x=24
#
#   Z = Hoehe. Alle Stuetzen gleich hoch (3.20m).
#   Traeger HEB 300/340 auf Stuetzenoberkante.
#   Platte auf Traegeroberkante.

stuetzen_grid = {
    "S1a": {"x": 0.0,  "y": 0.8},
    "S2a": {"x": 10.0, "y": 0.8},
    "S3a": {"x": 24.0, "y": 0.8},
    "S1b": {"x": 0.0,  "y": 3.2},
    "S2b": {"x": 10.0, "y": 3.2},
    "S3b": {"x": 24.0, "y": 3.2},
}

traeger_layout = {
    "T1a": {"von": "S1a", "bis": "S2a", "profil_h": 0.30},
    "T1b": {"von": "S1b", "bis": "S2b", "profil_h": 0.30},
    "T2a": {"von": "S2a", "bis": "S3a", "profil_h": 0.34},
    "T2b": {"von": "S2b", "bis": "S3b", "profil_h": 0.34},
}

z_stuetze_ok = 3.20
profil_h_max = 0.34
z_uk_platte = z_stuetze_ok + profil_h_max
platte_d = 0.25

fig = go.Figure()

# ============================================================
# Box-Mesh mit legendgroup
# ============================================================
def add_box(x0, x1, y0, y1, z0, z1, color, name, group,
            opacity=0.9, legend=True):
    vx = [x0,x1,x1,x0, x0,x1,x1,x0]
    vy = [y0,y0,y1,y1, y0,y0,y1,y1]
    vz = [z0,z0,z0,z0, z1,z1,z1,z1]
    ti = [0, 0, 4, 4, 0, 0, 1, 1, 0, 0, 3, 3]
    tj = [1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 2, 6]
    tk = [2, 3, 6, 7, 5, 4, 6, 5, 7, 4, 6, 7]
    fig.add_trace(go.Mesh3d(
        x=vx, y=vy, z=vz, i=ti, j=tj, k=tk,
        color=color, opacity=opacity, name=name,
        flatshading=True,
        showlegend=legend, legendgroup=group,
        hovertext=name, hoverinfo="text",
    ))

# ============================================================
# 6 Stuetzen
# ============================================================
first_s = True
for e in daten:
    if e["typ"] != "Stuetze": continue
    pos = stuetzen_grid[e["id"]]
    bm, hm = e["b_cm"]/100, e["h_cm"]/100
    info = f"{e['id']}: {e['b_cm']}x{e['h_cm']}cm, H={e['laenge_m']}m, {e['material']}"
    add_box(
        pos["x"] - bm/2, pos["x"] + bm/2,
        pos["y"] - hm/2, pos["y"] + hm/2,
        0, e["laenge_m"],
        "#78909C", "Stuetzen" if first_s else None,
        group="Stuetzen", legend=first_s
    )
    first_s = False

# ============================================================
# 4 Laengstraeger
# ============================================================
first_t = {"HEB 300": True, "HEB 340": True}
for e in daten:
    if e["typ"] != "Traeger": continue
    tl = traeger_layout[e["id"]]
    p1, p2 = stuetzen_grid[tl["von"]], stuetzen_grid[tl["bis"]]
    ph = tl["profil_h"]
    profil_name = "HEB 300" if ph < 0.32 else "HEB 340"
    color = "#F57C00" if profil_name == "HEB 300" else "#E65100"
    grp = f"Traeger {profil_name}"
    info = f"{e['id']}: {profil_name}, L={e['laenge_m']}m"

    add_box(
        min(p1["x"], p2["x"]), max(p1["x"], p2["x"]),
        p1["y"] - 0.075, p1["y"] + 0.075,
        z_stuetze_ok, z_stuetze_ok + ph,
        color, grp if first_t[profil_name] else None,
        group=grp, legend=first_t[profil_name]
    )
    first_t[profil_name] = False

# ============================================================
# Quertraeger
# ============================================================
first_q = True
for x_pos in [0.0, 10.0, 24.0]:
    add_box(
        x_pos - 0.10, x_pos + 0.10,
        0.8, 3.2,
        z_stuetze_ok, z_stuetze_ok + 0.25,
        "#FFA726", "Quertraeger" if first_q else None,
        group="Quertraeger", opacity=0.8, legend=first_q
    )
    first_q = False

# ============================================================
# Fahrbahnplatte
# ============================================================
platte = next(e for e in daten if e["typ"] == "Platte")
b = platte["b_cm"] / 100
add_box(
    -0.3, 24.3, 0.0, 4.0,
    z_uk_platte, z_uk_platte + platte_d,
    "#B0BEC5", "Fahrbahnplatte",
    group="Fahrbahnplatte", opacity=0.45
)

# ============================================================
# Gelaender (alle in einer legendgroup!)
# ============================================================
z_gel_ok = z_uk_platte + platte_d
gel_h = 1.1
gel_group = "Gelaender"

first_gel = True
for y_pos in [0.0, 4.0]:
    # Handlauf
    fig.add_trace(go.Scatter3d(
        x=[-0.3, 24.3], y=[y_pos, y_pos],
        z=[z_gel_ok + gel_h, z_gel_ok + gel_h],
        mode="lines", line=dict(color="#37474F", width=4),
        name="Gelaender" if first_gel else None,
        showlegend=first_gel, legendgroup=gel_group,
        hoverinfo="skip",
    ))
    first_gel = False
    # Pfosten
    for xp in np.arange(0, 24.5, 2.0):
        fig.add_trace(go.Scatter3d(
            x=[xp, xp], y=[y_pos, y_pos],
            z=[z_gel_ok, z_gel_ok + gel_h],
            mode="lines", line=dict(color="#37474F", width=2),
            showlegend=False, legendgroup=gel_group,
            hoverinfo="skip",
        ))

# ============================================================
# Spannweiten-Annotation
# ============================================================
z_ann = -0.3
for label, x0, x1 in [("Feld 1: 10m", 0, 10), ("Feld 2: 14m", 10, 24)]:
    fig.add_trace(go.Scatter3d(
        x=[x0+0.3, x1-0.3], y=[2, 2], z=[z_ann, z_ann],
        mode="lines+text", line=dict(color="#1565C0", width=3),
        text=[None, label], textposition="top center",
        textfont=dict(size=12, color="#1565C0"),
        showlegend=False, legendgroup="Annotation",
        hoverinfo="skip",
    ))

# ============================================================
# Boden
# ============================================================
add_box(-2, 26, -1, 5, -0.02, 0, "#EEEEEE", None,
        group="Boden", opacity=0.2, legend=False)

# ============================================================
# Layout
# ============================================================
fig.update_layout(
    title=dict(
        text="Velobruecke: 3D-Modell aus JSON-Daten"
             "<br><sup>6 Stuetzen, 4 Laengstraeger (HEB 300 + HEB 340),"
             " Quertraeger, Fahrbahnplatte, Gelaender</sup>",
        font=dict(size=15),
    ),
    scene=dict(
        xaxis_title="X [m]",
        yaxis_title="Y [m]",
        zaxis_title="Z [m]",
        aspectmode="data",
        camera=dict(
            eye=dict(x=1.6, y=1.4, z=0.6),
            up=dict(x=0, y=0, z=1),
        ),
    ),
    width=950, height=650,
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.85)",
        font=dict(size=11),
        # Klick auf Legende blendet die ganze Gruppe ein/aus
        groupclick="togglegroup",
    ),
    margin=dict(l=0, r=0, t=60, b=0),
)

fig.show()
print()
print("Klickt auf die Legende: Jede Gruppe wird komplett ein-/ausgeblendet.")
print("Feld 1: 10m (HEB 300), Feld 2: 14m (HEB 340). Alles aus derselben JSON-Datei!")


Klickt auf die Legende: Jede Gruppe wird komplett ein-/ausgeblendet.
Feld 1: 10m (HEB 300), Feld 2: 14m (HEB 340). Alles aus derselben JSON-Datei!


### Was gerade passiert ist

Ihr habt eine JSON-Datei mit 6 Elementen eingelesen und daraus ein **vollständiges 3D-Modell** generiert. Die Geometrie, die Materialien, die Abmessungen kommen alle aus den Daten.

Das ist im Kern das, was BIM-Software macht: strukturierte Daten → 3D-Visualisierung. Nur dass BIM-Software zusätzlich Normen, Detaillierung und Kollisionserkennung enthält.

In **Woche 6** baut ihr genau so ein Modell systematisch auf, mit Knoten, Kanten und Flächen als Python-Datenstruktur.

---
## Zusammenfassung

| Was | CSV | JSON |
|-----|-----|------|
| Einlesen | `csv.DictReader(f)` | `json.load(f)` |
| Ergebnis | Liste von Dicts | Liste von Dicts |
| Datentypen | Alles String | Automatisch korrekt |
| Schreiben | `csv.writer(f)` | `json.dump(daten, f)` |

**Am Ende stehen immer Python-Dictionaries.** Alles aus Woche 3 funktioniert genauso.

---
*Digitales Bauen 1 | BFH AHB | Louis Trümpler*